# **Set up**

In [17]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Sat Sep 19 07:22:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted


In [9]:
# ── CELL 3: Clone or pull the GitHub repo ─────────────────────────────────────
#
# ⚠️  CHANGE BRANCH_NAME BEFORE RUNNING
#     Examples:
#       'advisor_colab_experiments'   ← advisor testing
#       'pair-1/smollm2-1.7b'         ← student pair 1
#       'main'                        ← read-only reference (do not push to main)

import os, subprocess, sys
from google.colab import userdata

BRANCH_NAME = 'btt_setup_VD'
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR    = '/content/project'   # repo root

# ── Load PAT ──────────────────────────────────────────────────────────────────
try:
    PAT = userdata.get('GITHUB_PAT')
    assert PAT, 'Secret is empty'
    print(f'✅ GITHUB_PAT loaded ({len(PAT)} chars)')
except Exception as e:
    print(f'❌ GITHUB_PAT: {e}')
    print('   Open 🔑 Secrets → add GITHUB_PAT → toggle Notebook access ON')
    raise SystemExit('Cannot clone without GITHUB_PAT')

REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

def git(args, cwd=None, check=True):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f'❌ git error: {r.stderr.replace(PAT, "***").strip()}')
        raise RuntimeError(' '.join(args))
    return r.stdout.strip()

# ── Clone or pull ──────────────────────────────────────────────────────────────
if not os.path.exists(f'{REPO_DIR}/.git'):
    if os.path.exists(REPO_DIR):
        print('Removing broken directory ...')
        subprocess.run(['rm', '-rf', REPO_DIR])
    print(f'Cloning branch "{BRANCH_NAME}" ...')
    git(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, REPO_DIR])
    print(f'✅ Cloned to {REPO_DIR}')
else:
    print(f'Pulling latest from "{BRANCH_NAME}" ...')
    git(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git(['git', 'pull', '--rebase', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    print(f'✅ Up to date')

# ── Auto-detect where main.py lives (repo root or code/ subfolder) ────────────
if os.path.exists(f'{REPO_DIR}/main.py'):
    CODE_DIR = REPO_DIR
elif os.path.exists(f'{REPO_DIR}/code/main.py'):
    CODE_DIR = f'{REPO_DIR}/code'
else:
    CODE_DIR = None
    print('❌ Cannot find main.py — checked repo root and code/ subfolder')
    print(f'   Contents of {REPO_DIR}: {os.listdir(REPO_DIR)}')

if CODE_DIR:
    print(f'✅ Code directory: {CODE_DIR}')
    missing = [f for f in ['requirements.txt', 'requirements_colab.txt']
               if not os.path.exists(f'{CODE_DIR}/{f}')]
    if missing:
        print(f'⚠️  Missing in {CODE_DIR}: {missing}')
        print('   Push these files from your local machine, or run:')
        print('   !git -C /content/project fetch origin')
        print('   !git -C /content/project checkout origin/main -- code/requirements_colab.txt')
    else:
        print('✅ requirements.txt and requirements_colab.txt found')

    os.chdir(CODE_DIR)
    sys.path.insert(0, CODE_DIR)
    # Store CODE_DIR for other cells to use
    os.environ['SLM_CODE_DIR'] = CODE_DIR
    print(f'Working directory: {os.getcwd()}')

✅ GITHUB_PAT loaded (40 chars)
Pulling latest from "btt_setup_VD" ...
✅ Up to date
✅ Code directory: /content/project/code
✅ requirements.txt and requirements_colab.txt found
Working directory: /content/project/code


In [ ]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Reads requirements files from the code directory found in Cell 3.
# Takes 3–4 minutes. Normal to see some warnings.
import os, subprocess, sys

CODE_DIR = os.environ.get('SLM_CODE_DIR', '/content/project/code')
print(f'Installing from: {CODE_DIR}')

def pip_install(filename):
    path = f'{CODE_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'❌ {filename} not found at {path}')
        print('   Make sure Cell 3 ran successfully first.')
        return False
    print(f'\nInstalling {filename} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    tail = (result.stdout + result.stderr).strip().split('\n')
    for line in tail[-5:]:
        if line.strip():
            print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {filename}')
        return False
    print(f'✅ {filename} done')
    return True

ok1 = pip_install('requirements.txt')
ok2 = pip_install('requirements_colab.txt')
print('\n✅ All dependencies installed' if (ok1 and ok2) else '\n⚠️  Check errors above')

Installing from: /content/project/code

Installing requirements.txt ...
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.3 MB/s eta 0:00:00
✅ requirements.txt done

Installing requirements_colab.txt ...


In [ ]:
# ── CELL 5: Drive folder structure and HuggingFace cache ─────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
for d in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
          f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
          f'{DRIVE_ROOT}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

print(f'✅ Drive folders ready under {DRIVE_ROOT}')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

In [ ]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        print(f'  ⚠️  {name} is empty — {"add value in 🔑 Secrets" if required else "optional"}')
    except Exception:
        print(f'  ❌ {name} not found — {"open 🔑 Secrets and toggle Notebook access ON" if required else "optional"}')
    return not required

print('Loading secrets:')
all_ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])
print('\n✅ Required secrets loaded' if all_ok else '\n❌ Fix missing secrets before running the pipeline')

In [ ]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

code_dir  = os.environ.get('SLM_CODE_DIR', '')
drive_ok  = os.path.exists('/content/drive/MyDrive')
gpu_ok    = torch.cuda.is_available()
vram_ok   = gpu_ok and torch.cuda.get_device_properties(0).total_memory > 14e9

checks = [
    ('T4 GPU (≥14 GB VRAM)',   vram_ok),
    ('Drive mounted',          drive_ok),
    ('Code directory found',   bool(code_dir) and os.path.exists(code_dir)),
    ('main.py present',        os.path.exists(f'{code_dir}/main.py') if code_dir else False),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

print('Setup verification:')
print(f'  Code directory: {code_dir or "NOT SET"}')
print()
all_ok = True
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"} {label}')
    if not ok:
        all_ok = False

print()
print('🚀 Ready! Scroll down to run the pipeline.' if all_ok else
      '⚠️  Fix ❌ items before running the pipeline.')

# **Commit**

In [ ]:
# ── CELL 2: Reusable commit helper — run setup_repo() first, then call this
#            any time I want to push a file to my branch.
import subprocess
import os
import shutil

REPO = '/content/project'
BRANCH = 'btt_setup_VD'


def commit_file(source_path: str, dest_relative_path: str, commit_message: str,
                 under_code_dir: bool = True):
    """
    Copy a file into the repo and push it to your branch.

    source_path         — full path to the file right now (e.g. in Drive)
    dest_relative_path  — where it should live inside the repo (or code dir),
                           e.g. 'notebooks/fine_tuning_VD.ipynb'
    commit_message       — your commit message
    under_code_dir       — True if this path is relative to CODE_DIR (e.g.
                           notebooks/, configs/ — most things). False if it's
                           relative to the REPO root instead.
    """
    base_dir = CODE_DIR if under_code_dir else REPO
    dest_path = f'{base_dir}/{dest_relative_path}'
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    shutil.copy(source_path, dest_path)
    print(f"Copied to: {dest_path}")

    # git commands always run relative to REPO root, so the path passed to
    # `git add` needs to include the code/ prefix if that's where the file is
    git_relative_path = os.path.relpath(dest_path, REPO)

    for cmd in [
        ['git', 'checkout', BRANCH],
        ['git', 'add', git_relative_path],
        ['git', 'commit', '-m', commit_message],
        ['git', 'push', 'origin', BRANCH],
    ]:
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
        out = (r.stdout + r.stderr).strip()
        if out:
            print(out)

    print('\nDone. Verify at:')
    print(f'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/tree/{BRANCH}/{os.path.dirname(git_relative_path)}')


# ── Example usage ────────────────────────────────────────────────────────
# Every time you want to save a notebook (or any file) to your branch,
# just call this one line — no need to repeat the clone/copy/commit steps:

commit_file(
    source_path='/content/drive/MyDrive/Colab Notebooks/fine_tuning_VD.ipynb',
    dest_relative_path='notebooks/fine_tuning_VD.ipynb',
    commit_message='Update fine tuning notebook',
    under_code_dir=False,   # notebooks/ lives at repo root, not under code/
)


In [6]:
#When local changes open and can't commit
import subprocess

REPO_DIR = "/content/project"

print("1. Stashing local changes...")
subprocess.run(["git", "stash", "save", "colab_work_in_progress"], cwd=REPO_DIR, check=True)

print("2. Pulling latest commits from GitHub...")
subprocess.run(["git", "config", "pull.rebase", "true"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "btt_setup_VD"], cwd=REPO_DIR, check=True)

print("3. Restoring stashed local edits...")
pop_res = subprocess.run(["git", "stash", "pop"], cwd=REPO_DIR, capture_output=True, text=True)

if pop_res.returncode != 0:
    print("Conflict while restoring stash. Resolving by keeping your local changes:")
    subprocess.run(["git", "checkout", "--ours", "."], cwd=REPO_DIR, check=False)
    subprocess.run(["git", "add", "."], cwd=REPO_DIR, check=False)
    print("✔ Kept local versions.")
else:
    print("✔ Stash restored cleanly without conflicts.")

print("\n✔ Repository is fully synchronized and up to date!")

1. Stashing local changes...
2. Pulling latest commits from GitHub...
3. Restoring stashed local edits...
✔ Stash restored cleanly without conflicts.

✔ Repository is fully synchronized and up to date!


In [8]:
#hard reset command will completely resolve the divergent branch and uncommitted
# changes conflict by snapping your local Colab working tree directly to the exact
# state of origin/btt_setup_VD.
!cd /content/project && git fetch origin && git reset --hard origin/btt_setup_VD

HEAD is now at 167392a Add overview of experiments directory structure


# **Run Experiment**

In [ ]:
#PULL LATEST GIT
!cd /content/project && git pull --no-rebase #check latest GIT
# !cd /content/project && git push origin btt_setup_VD
# !cd /content/project && git status
# !cd /content/project && git checkout code/configs/phase1_config_vd.yaml && git pull #merge to github version
# !cd /content/project && git checkout code/phase1/evaluation/llm_judge.py && git pull
# !cd /content/project && git checkout code/phase1/labeling/frontier_llm.py && git pull #merge to github version

# **Main.py**

In [23]:
#FULL TRAINING main.py
# import os
# CODE = os.environ.get('SLM_CODE_DIR', '/content/project/code')
# !python "$CODE/main.py" \
#     --phase 1 \
#     --config "$CODE/configs/phase1_config_vd.yaml" \ #on cleaned data rn
#     --device_mode colab

01:10:36 | INFO     | __main__ | [main] Overriding device_mode: colab → colab
01:10:37 | INFO     | __main__ | [main] Colab Secrets loaded.
01:10:37 | INFO     | __main__ | 
  SLM Distillation — Phase 1 | Mode: train
  Config:      /content/project/code/configs/phase1_config_vd.yaml
  Device mode: colab
  Student SLM: HuggingFaceTB/SmolLM2-360M-Instruct
  Teacher LLM: claude-haiku-4-5
  Drive root:  /content/drive/MyDrive/slm-distillation
01:10:37 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
01:10:38 | INFO     | phase1.pipeline | [pipeline] Run ID : 20260916_0110_SmolLM2-360M-Instruct_ep3
01:10:38 | INFO     | phase1.pipeline | [pipeline] Outputs → /content/drive/MyDrive/slm-distillation/outputs/20260916_0110_SmolLM2-360M-Instruct_ep3
01:10:38 | INFO     | phase1.pipeline | 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 1: Clustering
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
01:10:39 | INFO     | datasets | TensorFlow version 2

# **# Run_experimet.py**

In [24]:
!python /content/project/code/run_experiments.py


[STARTING EXPERIMENT]: clean_baseline_default
[COMPLETE] 'clean_baseline_default' finished in 5.41 min
[CHECKPOINT] Progress saved to Drive: /content/drive/MyDrive/slm-distillation/experiment_comparison_clean.csv

[STARTING EXPERIMENT]: clean_lr_extreme_1e-3
[OPTIMIZATION] Reusing precomputed baseline evaluation.
[COMPLETE] 'clean_lr_extreme_1e-3' finished in 3.45 min
[CHECKPOINT] Progress saved to Drive: /content/drive/MyDrive/slm-distillation/experiment_comparison_clean.csv

[STARTING EXPERIMENT]: clean_lr_higher_8e-4
[OPTIMIZATION] Reusing precomputed baseline evaluation.
[COMPLETE] 'clean_lr_higher_8e-4' finished in 3.53 min
[CHECKPOINT] Progress saved to Drive: /content/drive/MyDrive/slm-distillation/experiment_comparison_clean.csv


EXPERIMENT SWEEP COMPLETE — SUMMARY TABLE
                 label                               model     lr  epochs  lora_r  base_cosine_sim  ft_cosine_sim  Δ_cosine_sim  judge_composite  elapsed_min  status
clean_baseline_default HuggingFaceTB/SmolL

# **LLama 3.2 3B**

In [ ]:
!python /content/project/code/run_experiments_llama.py

# **Unknown**

In [ ]:
import json
from pathlib import Path
import torch
from unsloth import FastLanguageModel

EXPORT_DIR = Path("/content/drive/MyDrive/slm-distillation/exported_models/best_smollm2_360m_lr_1e-3")
TEST_DATA_PATH = Path("/content/drive/MyDrive/slm-distillation/data/processed/test.jsonl")

# 1. Load via FastLanguageModel so patched methods match layer attributes
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(EXPORT_DIR),
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=False,
    device_map="cuda",
)
FastLanguageModel.for_inference(model)  # Attaches Unsloth fast inference kernels

# 2. Run the spot-check
delimiter = "<|im_start|>assistant\n"
samples_evaluated = 0

print("\n" + "=" * 95)
print(f"{'PROMPT ID':<10} | {'PREDICTED LABEL (SmolLM2-360M)':<40} | {'TEACHER TARGET (Claude Haiku)'}")
print("=" * 95)

if TEST_DATA_PATH.exists():
    with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if samples_evaluated >= 5:
                break
            record = json.loads(line)
            raw_text = record["text"]

            if delimiter not in raw_text:
                continue

            prompt, gold_label = raw_text.split(delimiter, 1)
            prompt = prompt + delimiter
            gold_label = gold_label.replace("<|im_end|>", "").strip()

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=25,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            pred = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            pid = record.get("prompt_id", "P?")
            print(f"{pid:<10} | {pred[:40]:<40} | {gold_label[:40]}")
            samples_evaluated += 1
print("=" * 95)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=25) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT ID  | PREDICTED LABEL (SmolLM2-360M)           | TEACHER TARGET (Claude Haiku)


Both `max_new_tokens` (=25) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

P2         | primary issue: shipment options          | Shipping and delivery options inquiry


Both `max_new_tokens` (=25) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


P3         | Cluster 1: Shipping Options              | Shipping and Delivery Options Inquiry


Both `max_new_tokens` (=25) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


P4         | Common theme: shipment options           | Customers requesting information about a
P5         | Cluster: "Shipping Options"              | Shipping and delivery methods inquiry


# **# Clean up uncessary file**

In [ ]:
import os
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/drive/MyDrive/slm-distillation")

# Target extensions and file patterns that are 100% temporary
SAFE_TO_DELETE_EXTS = {".arrow", ".log", ".bin"}
SAFE_TO_DELETE_NAMES = {"optimizer.pt", "rng_state.pth"}

freed_bytes = 0
deleted_count = 0

print(f"Scanning for temporary artifacts in {PROJECT_ROOT}...\n")

for root, dirs, files in os.walk(PROJECT_ROOT):
    # NEVER touch the final exported standalone model folder
    if "exported_models" in root:
        continue

    for file in files:
        fpath = Path(root) / file

        # Check if file matches temporary patterns
        should_delete = False
        if fpath.suffix in SAFE_TO_DELETE_EXTS:
            should_delete = True
        elif fpath.name in SAFE_TO_DELETE_NAMES:
            should_delete = True

        if should_delete:
            try:
                size = fpath.stat().st_size
                fpath.unlink()
                freed_bytes += size
                deleted_count += 1
            except Exception as e:
                print(f"Could not delete {fpath}: {e}")

freed_gb = freed_bytes / (1024 ** 3)
print("=" * 60)
print(f"Clean up complete!")
print(f"Files deleted: {deleted_count}")
print(f"Storage space reclaimed: {freed_gb:.2f} GB")
print("=" * 60)

Scanning for temporary artifacts in /content/drive/MyDrive/slm-distillation...

Clean up complete!
Files deleted: 422
Storage space reclaimed: 4.77 GB


In [ ]:
import os
from pathlib import Path
import re

DRIVE_ROOT = Path("/content/drive/MyDrive/slm-distillation")
WINNING_RUN_FOLDER = "20260913_0727_SmolLM2-360M-Instruct_ep3"
HASH_REGEX = re.compile(r"^[0-9a-f]{64}$")

deleted_files = 0
freed_bytes = 0

print("Scanning Drive for safe-to-delete files...\n")

for root, dirs, files in os.walk(DRIVE_ROOT):
    # NEVER touch the final standalone merged export
    if "exported_models" in root:
        continue

    for f in files:
        fpath = Path(root) / f
        should_delete = False

        # 1. Delete 64-char hash cache files
        if HASH_REGEX.match(f):
            should_delete = True

        # 2. Delete losing adapter checkpoints (keeping only the winning run)
        elif f == "adapter_model.safetensors":
            if WINNING_RUN_FOLDER not in root:
                should_delete = True

        # 3. Delete any intermediate base model weights outside exported_models
        elif f == "model.safetensors":
            should_delete = True

        if should_delete:
            try:
                size = fpath.stat().st_size
                fpath.unlink()
                freed_bytes += size
                deleted_files += 1
                print(f"Deleted: {fpath.relative_to(DRIVE_ROOT)}")
            except Exception as e:
                print(f"Error deleting {fpath}: {e}")

print("=" * 60)
print(f"Deleted {deleted_files} files.")
print(f"Reclaimed {freed_bytes / (1024**3):.2f} GB.")
print("=" * 60)

Scanning Drive for safe-to-delete files...

Deleted: outputs/20260901_2346_SmolLM2-360M-Instruct_ep3/models/lora_adapter/adapter_model.safetensors
Deleted: outputs/20260901_2346_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-5/adapter_model.safetensors
Deleted: outputs/20260901_2346_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-10/adapter_model.safetensors
Deleted: outputs/20260901_2346_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-15/adapter_model.safetensors
Deleted: outputs/20260913_0409_SmolLM2-360M-Instruct_ep3/models/lora_adapter/adapter_model.safetensors
Deleted: outputs/20260913_0409_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-5/adapter_model.safetensors
Deleted: outputs/20260913_0409_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-10/adapter_model.safetensors
Deleted: outputs/20260913_0409_SmolLM2-360M-Instruct_ep3/models/lora_adapter/checkpoint-15/adapter_model.safetensors
Deleted: outputs/20260913_0435_SmolLM2-360M-Instru